# Single spin: Bayesian optimization of a gate

This is similar to the O2B_Single_qubit_gate example, except that it uses Bayesian optimization <cite data-cite="shahriari2016taking">(Shahriari et al., 2016)</cite> instead of gradient descent.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from jax import Array

from paraqeet.eom.schroedinger_equation import SchroedingerEquation
from paraqeet.hamiltonian.drive import Drive
from paraqeet.hamiltonian.qubit import QubitHamiltonian
from paraqeet.logger import Logger
from paraqeet.measurement.unitary_fidelity import UnitaryFidelity
from paraqeet.optimization_map import OptimizationMap
from paraqeet.optimizers.bayesian_optimizer import BayesianOptimizer
from paraqeet.propagation.expm_goat import ExpmGOAT
from paraqeet.quantity import Quantity
from paraqeet.signal.envelopes import ConstantEnvelope
from paraqeet.signal.iq_mixer import IQMixer

## Setup

We first set up the qubit system we want to control. We set the qubit frequency $\omega_q / 2 \pi$ to be $4.8$ GHz and define the Hamiltonian as $$H(t)=H_\text{drift}+H_c(t)= \frac{\omega_q}{2} \sigma_z + \Omega(t)\sigma_x, $$ where $\Omega(t)$ will be supplied by the generator.

In [ ]:
freq_q = 4.8e9
omega_q = 2 * np.pi * freq_q

qubit_hamiltonian = QubitHamiltonian(
    frequency=Quantity(omega_q, 0.8 * omega_q, 1.2 * omega_q, unit="Hz", two_pi=True), drives=[]
)
model = SchroedingerEquation(
    hamiltonian_func=qubit_hamiltonian.get_value,
    hamiltonian_gradient_func=qubit_hamiltonian.get_gradient,
)

For signal generation, we define a simple cosine shaped tone generator $A \cos(\omega t)$

In [ ]:
t_simu = 3e-9
tone = ConstantEnvelope()
tone.t_final.set_value(t_simu)
gen = IQMixer(envelopes=[tone])

We can inspect the pre-defined parameters with

In [ ]:
params_gen = gen.get_parameters()

In this notebook, we would like to optimize the amplitude `Amplitude` and frequency `lo_freq` if the drive. We add a drive on the qubit.

In [ ]:
freq = 4.8e9 * 2 * np.pi
sigma_x = qubit_hamiltonian.sigma_x
drive = Drive(sigma_x, gen)
qubit_hamiltonian.drives = [drive]
model = SchroedingerEquation(
    hamiltonian_func=qubit_hamiltonian.get_value,
    hamiltonian_gradient_func=qubit_hamiltonian.get_gradient,
)

Textbook values for implementing an $X$ rotation on this system at a time $T$ would be $\omega=\omega_q$ and $A=\pi/T$. We use some offset from these values as initial guess to demonstrate the optimization procedure.

In [ ]:
params_gen[0].set_value(0.5 * np.pi / t_simu)
params_gen[2].set_value(1.01 * freq)

We select a propagation method, piecewise constant exponentiation, and configure an $X$-gate as a target gate. Also we initialize the identity at time $0$.

In [ ]:
times = np.array([0.0, t_simu])

prop = ExpmGOAT(
    eom_func=model.get_value,
    eom_gradient_func=model.get_gradient,
    resolution=100e9,
    initial_state=np.identity(2),
)

gate_fid = UnitaryFidelity(propagation_func=prop.get_value, propagation_gradient_func=prop.get_gradient, gate=sigma_x)

In [ ]:
from plotting import plot_signal_and_dynamics

ts = np.linspace(0.0, t_simu, 301)
plot_signal_and_dynamics(gen, prop, ts, state_labels=[r"$|0\rangle$", r"$|1\rangle$"])

As expected, we get a partial transfer and a low fidelity.

In [ ]:
print(f"Gate fidelity: {gate_fid.get_value(times)}")

## Custom logger implementation
We define an optimizer and link our fidelity measure as a goal function and the parameters of the cosine tone. We also use a custom logger class to collect all samples that the optimizer takes

In [ ]:
samples = []


class CustomLogger(Logger):
    """Custom logger class definition."""

    def log(self, params: list[Quantity], infid: Array):
        """Log the list of quantities and the fidelity.

        Parameters
        ----------
        params: list[Quantity]
            List of parameters of the system.
        infid: Array
            Inverse of fidelity.

        """
        samples.append((params[0].get_value(), params[1].get_value()))


optmap = OptimizationMap()
optmap.add(gen, [params_gen[0], params_gen[2], params_gen[3]])
opt = BayesianOptimizer(measure_func=gate_fid.get_value, optimization_map=optmap, initial_samples=10, iterations=100)
opt.logger = CustomLogger()

In [ ]:
opt.optimize(times)

The plot shows the all the samples that the optimization took in the two-dimensional parameter space. The red dot marks the best value.

In [ ]:
plt.figure(figsize=(4, 4))
plt.scatter([s[0] for s in samples[:-1]], [s[1] for s in samples[:-1]], c="blue")
plt.scatter([params_gen[0].get_value()], [params_gen[2].get_value()], c="red", marker="o", s=100)
plt.xlim(params_gen[0].get_min_value()[0], params_gen[0].get_max_value()[0])
plt.ylim(params_gen[2].get_min_value()[0], params_gen[2].get_max_value()[0])
plt.xlabel(params_gen[0].get_name())
plt.ylabel(params_gen[2].get_name())
plt.show()

In [ ]:
plot_signal_and_dynamics(gen, prop, ts, state_labels=[r"$|0\rangle$", r"$|1\rangle$"])

We can see from the plot and optimizer output that we have found good controls.

In [ ]:
print(f"Gate fidelity: {gate_fid.get_value(times)}")

## References

- **(Shahriari et al., 2016)** B. Shahriari et al., "Taking the human out of the loop: A review of Bayesian optimization," *Proceedings of the IEEE* **104**, 148–175 (2016).